In [ ]:
# =============================================================================
# EXPERIMENT: XGBoost with conservative defaults to check if good dataset    on preprocessed 6 data    nieuwe tips guilem
# Description: Use Optuna to find optimal XGBoost parameters
# Expected runtime: 1-2 hours   
# =============================================================================

In [2]:
# =============================================================================
# 2. DEFINE EXPERIMENT PARAMETERS
# =============================================================================
EXPERIMENT_NAME = "xgboost_optuna"
MODEL_DESCRIPTION = "XGBoost with Bayesian Hyperparameter Optimization on preprocessed 4 data"

In [ ]:
# =============================================================================
# QUICK BASELINE: XGBoost with Conservative Defaults
# Expected runtime: ~5 minutes
# =============================================================================

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score
from datetime import datetime

print("="*80)
print("XGBOOST BASELINE TEST")
print("="*80)

# =============================================================================
# 1. LOAD DATA
# =============================================================================

print("Loading preprocessed data...")
X = pd.read_pickle('../../data/processed/X_train_processed.pkl')
y = pd.read_pickle('../../data/processed/y_train.pkl')
X_test = pd.read_pickle('../../data/processed/X_test_processed.pkl')
test_ids = pd.read_pickle('../../data/processed/test_ids.pkl')

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

print(f"X_train: {X.shape}")
print(f"y_train: {y.shape}")
print(f"X_test: {X_test.shape}")

# Class imbalance
neg_samples = (y == 0).sum()
pos_samples = (y == 1).sum()
scale_pos_weight = neg_samples / pos_samples

print(f"\nClass distribution:")
print(f"  Negative (0): {neg_samples:,} ({neg_samples/len(y)*100:.1f}%)")
print(f"  Positive (1): {pos_samples:,} ({pos_samples/len(y)*100:.1f}%)")
print(f"  Scale pos weight: {scale_pos_weight:.2f}")

# =============================================================================
# 2. BASELINE MODEL - CONSERVATIVE PARAMETERS
# =============================================================================

print("\n" + "="*80)
print("TRAINING BASELINE XGBOOST")
print("="*80)

# Conservative parameters - proven to work well
baseline_params = {
    'n_estimators': 300,
    'learning_rate': 0.05,
    'max_depth': 4,
    'min_child_weight': 3,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'scale_pos_weight': scale_pos_weight,
    'random_state': 42,
    'tree_method': 'hist',  # Faster
    'eval_metric': 'auc'
}

print("\nBaseline parameters:")
for param, value in baseline_params.items():
    if param != 'scale_pos_weight':
        print(f"  {param}: {value}")
    else:
        print(f"  {param}: {value:.2f}")

model = xgb.XGBClassifier(**baseline_params)

# =============================================================================
# 3. CROSS-VALIDATION
# =============================================================================

print("\n--- Running 5-Fold Cross-Validation ---")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    model, X, y, 
    cv=cv, 
    scoring='roc_auc', 
    n_jobs=-1,
    verbose=0
)

print(f"\nCV Results:")
print(f"  Fold scores: {[f'{s:.4f}' for s in cv_scores]}")
print(f"  Mean CV ROC-AUC: {cv_scores.mean():.4f}")
print(f"  Std CV ROC-AUC: {cv_scores.std():.4f}")

# =============================================================================
# 4. TRAIN ON FULL DATA
# =============================================================================

print("\n--- Training on full dataset ---")

model.fit(X, y, verbose=False)

# Training performance
train_proba = model.predict_proba(X)[:, 1]
train_auc = roc_auc_score(y, train_proba)

print(f"\nTraining ROC-AUC: {train_auc:.4f}")
print(f"CV ROC-AUC: {cv_scores.mean():.4f}")

gap = train_auc - cv_scores.mean()
print(f"Overfitting gap: {gap:.4f}")

if gap < 0.05:
    print("  ✓ Excellent! Very little overfitting")
elif gap < 0.10:
    print("  ✓ Good! Acceptable overfitting")
elif gap < 0.15:
    print("  ⚠️ Moderate overfitting - Optuna should help")
else:
    print("  ❌ High overfitting - need more regularization")

# =============================================================================
# 5. GENERATE TEST PREDICTIONS
# =============================================================================

print("\n--- Generating test predictions ---")

test_proba = model.predict_proba(X_test)[:, 1]

print(f"\nTest prediction statistics:")
print(f"  Min: {test_proba.min():.4f}")
print(f"  Max: {test_proba.max():.4f}")
print(f"  Mean: {test_proba.mean():.4f}")
print(f"  Median: {np.median(test_proba):.4f}")

# Distribution
print(f"\nPrediction distribution:")
bins = [0, 0.05, 0.10, 0.15, 0.20, 0.30, 1.0]
for i in range(len(bins)-1):
    count = ((test_proba >= bins[i]) & (test_proba < bins[i+1])).sum()
    print(f"  {bins[i]:.2f}-{bins[i+1]:.2f}: {count:4d} ({count/len(test_proba)*100:5.1f}%)")

# =============================================================================
# 6. SAVE BASELINE SUBMISSION
# =============================================================================

# =============================================================================
# 5. CREATE SUBMISSION FILE
# =============================================================================
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
submission_filename = f"../../outputs/predictions/XG_boost/{EXPERIMENT_NAME}_{timestamp}.csv"

submission = pd.DataFrame({
    'icustay_id': test_ids,
    'prediction': test_proba
})
submission.to_csv(submission_filename, index=False)
print(f"\n✓ Submission saved: {submission_filename}")

# =============================================================================
# 7. FEATURE IMPORTANCE (TOP 20)
# =============================================================================

print("\n" + "="*80)
print("TOP 20 MOST IMPORTANT FEATURES")
print("="*80)

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n" + feature_importance.head(20).to_string(index=False))

# =============================================================================
# 8. VERDICT
# =============================================================================

print("\n" + "="*80)
print("BASELINE VERDICT")
print("="*80)

print(f"\nBaseline XGBoost Performance:")
print(f"  CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"  Expected Kaggle: ~{cv_scores.mean() - 0.09:.3f} - {cv_scores.mean() - 0.05:.3f}")

# Compare to current best
current_best = 0.798914
expected_kaggle = cv_scores.mean() - 0.07  # Conservative estimate

print(f"\n  Current best Kaggle: {current_best:.4f}")
print(f"  Expected with baseline: ~{expected_kaggle:.4f}")

if expected_kaggle > current_best + 0.01:
    print(f"\n✅ EXCELLENT! Baseline already beats current best!")
    print(f"   Upload {submission_file} to Kaggle")
    print(f"   Then run Optuna to push even higher!")
elif expected_kaggle > current_best - 0.01:
    print(f"\n✅ GOOD! Baseline matches current best")
    print(f"   Optuna tuning should push you higher!")
else:
    print(f"\n⚠️ Baseline underperforms")
    print(f"   But Optuna might still find better params")

print(f"\n--- Recommendation ---")

if cv_scores.mean() > 0.88:
    print("✅ CV > 0.88 - XGBoost looks great!")
    print("   Proceed with Optuna (100 trials, ~2 hours)")
elif cv_scores.mean() > 0.85:
    print("✅ CV > 0.85 - XGBoost is working well")
    print("   Optuna should improve it significantly")
elif cv_scores.mean() > 0.82:
    print("⚠️ CV > 0.82 - Decent but not great")
    print("   Optuna might help, or try ensemble")
else:
    print("❌ CV < 0.82 - Something might be wrong")
    print("   Check data or try different model")

print("\n" + "="*80)
print("BASELINE TEST COMPLETE")
print("="*80)

XGBOOST BASELINE TEST
Loading preprocessed data...
X shape: (20885, 95)
y shape: (20885,)
X_train: (20885, 95)
y_train: (20885,)
X_test: (5221, 95)

Class distribution:
  Negative (0): 18,540 (88.8%)
  Positive (1): 2,345 (11.2%)
  Scale pos weight: 7.91

TRAINING BASELINE XGBOOST

Baseline parameters:
  n_estimators: 300
  learning_rate: 0.05
  max_depth: 4
  min_child_weight: 3
  subsample: 0.8
  colsample_bytree: 0.8
  reg_alpha: 0.1
  reg_lambda: 1.0
  scale_pos_weight: 7.91
  random_state: 42
  tree_method: hist
  eval_metric: auc

--- Running 5-Fold Cross-Validation ---

CV Results:
  Fold scores: ['0.8942', '0.9031', '0.9034', '0.9187', '0.8963']
  Mean CV ROC-AUC: 0.9031
  Std CV ROC-AUC: 0.0086

--- Training on full dataset ---

Training ROC-AUC: 0.9535
CV ROC-AUC: 0.9031
Overfitting gap: 0.0504
  ✓ Good! Acceptable overfitting

--- Generating test predictions ---

Test prediction statistics:
  Min: 0.0004
  Max: 0.9989
  Mean: 0.2930
  Median: 0.1825

Prediction distribution:

NameError: name 'submission_file' is not defined